# BRIDGE: three-class QLoRA fine-tuning

Train answer-only LoRA adapters using the revised prompt, with the original pinned Qwen2.5-3B weights quantized to NF4. Validation macro-F1 from full-label candidate scoring selects the best epoch. Test data is never loaded. Setup and data checks run locally; model loading and training require one allocated DeltaAI CUDA GPU. Upload both shared Python modules with this notebook. Run All launches training and final validation.


In [ ]:
from scripts.bridge_environment import disable_transformer_engine
disable_transformer_engine()

from pathlib import Path
from functools import partial
from datetime import datetime, timezone
import importlib.metadata as metadata
import json
import platform
import socket
import os

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from scripts.bridge_helpers import (
    LABELS, REVISED_SYSTEM_PROMPT, sha256, make_candidate_ids,
    make_prompt_encoder, predict as score_candidates,
    save_evaluation as save_run_evaluation,
)

ROOT = Path.cwd()
assert (ROOT / "data/processed/manifest.json").is_file(), "Open the notebook from IT299."
print("Host:", socket.gethostname())
print("Python:", platform.python_version())
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("BF16 supported:", torch.cuda.is_bf16_supported())

versions = {}
for name in ["torch", "transformers", "peft", "accelerate", "bitsandbytes",
             "datasets", "pandas", "numpy", "scikit-learn"]:
    try:
        versions[name] = metadata.version(name)
    except metadata.PackageNotFoundError:
        versions[name] = "not installed locally"
print(json.dumps(versions, indent=2))


## Training configuration

This experiment uses learning rate 1e-4 and microbatch size 4 with 4 gradient accumulation steps, preserving effective batch size 16 on one GPU. LoRA rank remains 8 and alpha remains 32. Start a fresh run for these changed settings; do not resume the earlier 2e-4 run. Checkpoints and evaluation outputs use a unique run ID. Three epochs are planned; full validation candidate scoring selects the best epoch. Recovery checkpoints are also saved every 100 optimizer updates. Resume explicitly from a checkpoint with a completion marker; work since that checkpoint must be repeated. The prior microbatch-1, learning-rate-2e-4 job took about five hours; throughput and memory use for this configuration still need to be measured.


In [ ]:
MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
SEED = 42
MAX_INPUT_TOKENS = 4096
SYSTEM_PROMPT = REVISED_SYSTEM_PROMPT

lock_path = ROOT / "results/bridge_baseline/model_revision.json"
assert lock_path.is_file(), "Copy the original baseline model_revision.json."
lock = json.loads(lock_path.read_text())
assert lock["model_id"] == MODEL_ID
MODEL_REVISION = lock["revision"]

torch.manual_seed(SEED)
np.random.seed(SEED)
RUN_ID = os.environ.get("BRIDGE_RUN_ID") or datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
assert RUN_ID and all(ch.isalnum() or ch in "_-" for ch in RUN_ID)
RESUME_CHECKPOINT = os.environ.get("BRIDGE_RESUME_CHECKPOINT")
RECOVERY_SAVE_STEPS = 100
STORAGE_ROOT = Path(os.environ.get("BRIDGE_STORAGE_ROOT", str(ROOT))).expanduser().resolve()
CHECKPOINT_DIR = STORAGE_ROOT / "checkpoints/bridge_qlora" / RUN_ID
RESULTS_DIR = STORAGE_ROOT / "results/bridge_qlora" / RUN_ID

TRAINING_CONFIG = {
    "learning_rate": 1e-4,
    "num_train_epochs": 3,
    "per_device_train_batch_size": 4,
    "gradient_accumulation_steps": 4,
    "warmup_ratio": 0.03,
    "weight_decay": 0.0,
    "lr_scheduler_type": "cosine",
    "gradient_checkpointing": False,
    "gradient_checkpointing_kwargs": {"use_reentrant": False},
    "optim": "adamw_torch",
    "adam_beta1": 0.9,
    "adam_beta2": 0.999,
    "adam_epsilon": 1e-8,
    "max_grad_norm": 1.0,
    "logging_steps": 20,
    "logging_strategy": "steps",
    "per_device_eval_batch_size": 1,
    "dataloader_num_workers": 0,
    "label_names": ["labels"],
    "bf16": True,
    "eval_strategy": "epoch",
    "save_strategy": "epoch",
    "save_total_limit": 2,
    "load_best_model_at_end": True,
    "metric_for_best_model": "macro_f1",
    "greater_is_better": True,
    "seed": SEED,
    "data_seed": SEED,
    "report_to": "none",
}
LORA_CONFIG = {
    "r": 8,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    "bias": "none",
    "task_type": "CAUSAL_LM",
    "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj",
                       "gate_proj", "up_proj", "down_proj"],
}
print("Model revision:", MODEL_REVISION)
print("Checkpoint directory:", CHECKPOINT_DIR)
print("Results directory:", RESULTS_DIR)
print("Effective batch size on one GPU:",
      TRAINING_CONFIG["per_device_train_batch_size"] * TRAINING_CONFIG["gradient_accumulation_steps"])
print(json.dumps({"training": TRAINING_CONFIG, "lora": LORA_CONFIG}, indent=2))


## Training and validation data

Both CSVs must match the saved cleaning manifest. Checks verify split names, row counts, labels, nonempty text, unique sentence IDs, and separation of sentence IDs and duplicate groups. No test examples are loaded. All training rows are retained; there is no oversampling or demonstration selection.


In [ ]:
manifest_path = ROOT / "data/processed/manifest.json"
manifest = json.loads(manifest_path.read_text())

def load_verified_split(name):
    assert name in {"train", "validation"}, "Only development splits are loaded here."
    path = ROOT / "data/processed" / f"{name}.csv"
    assert sha256(path) == manifest["output_sha256"][path.name], f"{name} checksum mismatch."
    frame = pd.read_csv(path, dtype=str, keep_default_na=False)
    assert {"sentence_id", "group_id", "text", "bias_label", "split"}.issubset(frame.columns)
    assert set(frame["split"]) == {name}
    assert len(frame) == manifest["split_rows"][name]
    assert set(frame["bias_label"]) == set(LABELS)
    assert frame["sentence_id"].is_unique
    assert frame["sentence_id"].str.strip().ne("").all()
    assert frame["group_id"].str.strip().ne("").all()
    assert frame["text"].str.strip().ne("").all()
    return frame

train = load_verified_split("train")
validation = load_verified_split("validation")
for column in ["sentence_id", "group_id"]:
    assert set(train[column]).isdisjoint(validation[column]), f"Overlap in {column}."

label_counts = pd.DataFrame({
    "train": train["bias_label"].value_counts(),
    "validation": validation["bias_label"].value_counts(),
}).reindex(LABELS)
display(label_counts)
print("Training rows:", len(train))
print("Validation rows:", len(validation))
DATA_PROVENANCE = {
    "dataset_revision": manifest["revision"],
    "manifest_sha256": sha256(manifest_path),
    "train_sha256": manifest["output_sha256"]["train.csv"],
    "validation_sha256": manifest["output_sha256"]["validation.csv"],
}

resume_config = None
if RESUME_CHECKPOINT:
    from scripts.bridge_batch import validate_checkpoint
    checkpoint = validate_checkpoint(Path(RESUME_CHECKPOINT))
    assert checkpoint.parent.resolve() == CHECKPOINT_DIR.resolve()
    resume_config = json.loads((RESULTS_DIR / "run_config.json").read_text())
    assert resume_config["status"] != "complete", "This run is already complete."
    for key, value in {"model": lock, "system_prompt": SYSTEM_PROMPT, "lora": LORA_CONFIG,
                       "max_input_tokens": MAX_INPUT_TOKENS, **DATA_PROVENANCE}.items():
        assert resume_config[key] == value, f"Resume mismatch: {key}"
    for key, value in TRAINING_CONFIG.items():
        assert resume_config["training_config"][key] == value, f"Resume setting changed: {key}"


## Tokenizer, answer-only examples, and padding

Prompts match revised zero-shot evaluation. Append the exact label-plus-EOS candidate used in scoring. Prompt and padding labels are -100; only answer tokens contribute to loss. No examples are truncated or packed.


In [ ]:
from transformers import AutoTokenizer
from datasets import Dataset
from scripts.bridge_training import make_training_example, AnswerOnlyCollator

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REVISION)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
encode_excerpt = make_prompt_encoder(tokenizer, system_prompt=SYSTEM_PROMPT)
candidate_ids = make_candidate_ids(tokenizer)
MAX_SEQUENCE_TOKENS = MAX_INPUT_TOKENS + max(map(len, candidate_ids.values()))

def tokenize_frame(frame):
    return Dataset.from_list([
        make_training_example(row.text, row.bias_label, prompt_encoder=encode_excerpt,
                              candidate_ids=candidate_ids, max_input_tokens=MAX_INPUT_TOKENS)
        for row in frame.itertuples(index=False)
    ])

train_dataset = tokenize_frame(train)
validation_dataset = tokenize_frame(validation)
for name, dataset in [("train", train_dataset), ("validation", validation_dataset)]:
    lengths = [len(row["input_ids"]) for row in dataset]
    assert max(lengths) <= MAX_SEQUENCE_TOKENS
    print(name, pd.Series(lengths).describe(percentiles=[0.5, 0.95, 0.99]))
collator = AnswerOnlyCollator(tokenizer.pad_token_id)
for i in range(3):
    row = train_dataset[i]
    target = [token for token in row["labels"] if token != -100]
    assert target == candidate_ids[train.iloc[i]["bias_label"]]
    print("Training answer:", tokenizer.decode(target))


## Quantized model and trainable adapters

Only LoRA weights are trainable. Gradient checkpointing is disabled to avoid activation recomputation. Monitor peak GPU memory during the first training run; enable it if activation memory is too high. KV caching is disabled.


In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from scripts.bridge_helpers import scores
import time

assert torch.cuda.is_available() and torch.cuda.is_bf16_supported(), "Use the DeltaAI PyTorch GPU kernel."
assert torch.cuda.device_count() == 1, "This notebook supports one visible GPU; restrict CUDA_VISIBLE_DEVICES before starting the kernel."
quantization = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, revision=MODEL_REVISION, quantization_config=quantization,
    dtype=torch.bfloat16, device_map={"": torch.cuda.current_device()}, attn_implementation="sdpa",
)
assert MAX_SEQUENCE_TOKENS <= model.config.max_position_embeddings
model = prepare_model_for_kbit_training(
    model, use_gradient_checkpointing=TRAINING_CONFIG["gradient_checkpointing"],
    gradient_checkpointing_kwargs=TRAINING_CONFIG["gradient_checkpointing_kwargs"],
)
model = get_peft_model(model, LoraConfig(**LORA_CONFIG))
model.config.use_cache = False
model.print_trainable_parameters()
assert all("lora_" in name for name, parameter in model.named_parameters() if parameter.requires_grad)
assert any(parameter.requires_grad for parameter in model.parameters())


## Validation-aware trainer

Each epoch is evaluated using the same three candidate sequences as the baseline, not token-level accuracy or validation loss. The custom evaluate method logs eval_macro_f1 and invokes evaluation callbacks so Trainer can select and reload the best checkpoint. Validation predictions are saved per training step. Training mode is restored after each evaluation.


In [ ]:
from transformers import TrainerCallback

class RecoveryCheckpointCallback(TrainerCallback):
    def on_step_end(self, args, state, control, **kwargs):
        if state.global_step % RECOVERY_SAVE_STEPS == 0:
            control.should_save = True
        return control

    def on_save(self, args, state, control, **kwargs):
        # Written only after Trainer has saved model, optimizer, scheduler and RNG.
        path = Path(args.output_dir) / f"checkpoint-{state.global_step}"
        (path / "checkpoint_complete.json").write_text(json.dumps({"global_step": state.global_step}) + "\n")
        return control

class BridgeTrainer(Trainer):
    def evaluate(self, eval_dataset=None, ignore_keys=None, metric_key_prefix="eval"):
        if eval_dataset is not None and eval_dataset is not self.eval_dataset:
            raise ValueError("This evaluator is fixed to the verified validation split.")
        was_training = self.model.training
        directory = RESULTS_DIR / f"validation_step_{self.state.global_step:06d}"
        directory.mkdir(parents=True, exist_ok=True)
        rows = []
        try:
            for i, example in enumerate(validation.to_dict("records")):
                prediction = score_candidates(
                    example["text"], model=self.model, candidate_ids=candidate_ids,
                    prompt_encoder=encode_excerpt, max_input_tokens=MAX_INPUT_TOKENS,
                )
                row = {**example, **prediction}
                rows.append(row)
                pd.DataFrame([row]).to_csv(directory / "predictions.csv", mode="w" if i == 0 else "a", header=i == 0, index=False)
                if (i + 1) % 100 == 0:
                    print(f"Validation: {i + 1}/{len(validation)}", flush=True)
            metrics, _ = scores(pd.DataFrame(rows))
            metrics = {f"{metric_key_prefix}_{key}": value for key, value in metrics.items()}
            (directory / "metrics.json").write_text(json.dumps(metrics, indent=2) + "\n")
            self.log(metrics)
            self.control = self.callback_handler.on_evaluate(self.args, self.state, self.control, metrics)
            return metrics
        finally:
            self.model.train(was_training)

training_args = TrainingArguments(output_dir=str(CHECKPOINT_DIR), **TRAINING_CONFIG)
trainer = BridgeTrainer(
    model=model, args=training_args, train_dataset=train_dataset,
    eval_dataset=validation_dataset, data_collator=collator, processing_class=tokenizer,
    callbacks=[RecoveryCheckpointCallback()],
)
RESULTS_DIR.mkdir(parents=True, exist_ok=bool(RESUME_CHECKPOINT))
run_config = {
    "condition": "small_qlora_revised_definition", "status": "training",
    "model": lock, **DATA_PROVENANCE, "system_prompt": SYSTEM_PROMPT,
    "training_config": TRAINING_CONFIG, "recovery_save_steps": RECOVERY_SAVE_STEPS,
    "resumed_from": RESUME_CHECKPOINT,
    "slurm_job_id": os.environ.get("SLURM_JOB_ID"),
    "training_args": training_args.to_dict(), "lora": LORA_CONFIG,
    "versions": versions, "python": platform.python_version(),
    "seed": SEED, "split": "validation", "n": len(validation),
    "max_input_tokens": MAX_INPUT_TOKENS, "max_sequence_tokens": MAX_SEQUENCE_TOKENS,
    "precision": "4-bit NF4 + double quantization, BF16 compute", "attention": "sdpa",
    "prediction_method": "candidate_label_scoring", "candidate_token_ids": candidate_ids,
    "answer_end_token_id": tokenizer.eos_token_id,
    "score_rule": "sum of conditional log probabilities, including answer end",
    "normalization": "softmax over the three candidate scores", "length_normalization": False,
    "candidate_forward_passes_per_example": 3, "batch_size": 1,
    "loss_mask": "prompt and padding ignored; answer including EOS supervised",
    "gpu": torch.cuda.get_device_name(0), "checkpoint_dir": str(CHECKPOINT_DIR),
}
if resume_config:
    attempts = resume_config.get("previous_attempts", []) + [{
        "slurm_job_id": resume_config.get("slurm_job_id"),
        "status": resume_config.get("status"),
        "resumed_from": resume_config.get("resumed_from"),
    }]
    run_config["previous_attempts"] = attempts
(RESULTS_DIR / "run_config.json").write_text(json.dumps(run_config, indent=2) + "\n")
print("Training outputs:", CHECKPOINT_DIR)


## Train and save the best adapter

This cell starts GPU training. Epoch and periodic recovery checkpoints include optimizer, scheduler, and RNG state. Epoch validation still determines the best model. The final adapter directory contains the best validation-macro-F1 adapter and tokenizer; base weights remain in the pinned model repository.


In [ ]:
started = time.perf_counter()
model.train()
try:
    train_output = trainer.train(resume_from_checkpoint=RESUME_CHECKPOINT)
except Exception:
    run_config["status"] = "interrupted_or_failed"
    (RESULTS_DIR / "run_config.json").write_text(json.dumps(run_config, indent=2) + "\n")
    raise
run_config["training_wall_seconds_including_validation"] = time.perf_counter() - started
run_config["best_checkpoint"] = trainer.state.best_model_checkpoint
run_config["best_validation_macro_f1"] = trainer.state.best_metric
run_config["status"] = "trained"
ADAPTER_DIR = CHECKPOINT_DIR / "best_adapter"
trainer.save_model(str(ADAPTER_DIR))
tokenizer.save_pretrained(ADAPTER_DIR)
trainer.save_state()
trainer.save_metrics("train", train_output.metrics)
run_config["adapter_dir"] = str(ADAPTER_DIR)
(RESULTS_DIR / "run_config.json").write_text(json.dumps(run_config, indent=2) + "\n")
print("Best adapter saved:", ADAPTER_DIR)


## Final validation of the selected adapter

Rerun candidate scoring for the selected checkpoint with three warm-up examples and fresh peak-memory statistics. Save the same prediction columns, per-class metrics, confusion matrix, and subgroup metrics as the baseline. No test evaluation occurs.


In [ ]:
model = trainer.model
model.eval()
model.gradient_checkpointing_disable()
for text in validation["text"].head(3):
    score_candidates(text, model=model, candidate_ids=candidate_ids,
                     prompt_encoder=encode_excerpt, max_input_tokens=MAX_INPUT_TOKENS)
torch.cuda.reset_peak_memory_stats(model.device)
run_config["status"] = "evaluating"
(RESULTS_DIR / "run_config.json").write_text(json.dumps(run_config, indent=2) + "\n")
validation[["sentence_id", "group_id"]].to_csv(RESULTS_DIR / "validation_sample_ids.csv", index=False)
rows = []
for i, example in enumerate(validation.to_dict("records")):
    row = {**example, **score_candidates(example["text"], model=model,
           candidate_ids=candidate_ids, prompt_encoder=encode_excerpt, max_input_tokens=MAX_INPUT_TOKENS)}
    rows.append(row)
    pd.DataFrame([row]).to_csv(RESULTS_DIR / "predictions.csv", mode="w" if i == 0 else "a", header=i == 0, index=False)
    if (i + 1) % 20 == 0 or i + 1 == len(validation):
        print(f"Completed {i + 1}/{len(validation)}", flush=True)
metrics = save_run_evaluation(pd.DataFrame(rows), RESULTS_DIR, run_config,
                              train_label_counts=manifest["label_counts"]["train"], device=model.device)
print("Saved:", RESULTS_DIR)
